# Basic Assertion Classification with cwyde

[pyConTextNLP](https://github.com/chapmanbe/pyConTextNLP) introduced the concept of
_context-based assertion classification_ for clinical NLP: a finding such as "pulmonary
embolism" carries very different meaning depending on whether it is negated, probable,
historical, or under investigation.

cwyde is the modern successor to pyConTextNLP. It is built on top of
[medspaCy](https://github.com/medspacy/medspacy) and adds:

1. A richer **assertion taxonomy** (`AssertionCategory`) that preserves clinically
   meaningful distinctions that medspaCy's flat booleans collapse.
2. The **INDICATION** category — a first-class representation for findings under
   investigation ("rule out X"), which is categorically different from negation.
3. **Formal modal semantics** via [gamen-hs](https://github.com/chapmanbe/gamen-hs):
   each assertion category is mapped to a doxastic formula that supports downstream
   reasoning (consistency checking, guideline validation).

This notebook mirrors the structure of the pyConTextNLP
[BasicSentenceMarkup](../../../pyConTextNLP/notebooks/BasicSentenceMarkup.ipynb)
demonstration, showing how the same clinical sentences are handled at each layer.

## 1. Loading the pipeline

In [1]:
from loguru import logger
logger.disable("PyRuSH")  # suppress PyRuSH per-token debug lines

import cwyde

nlp = cwyde.load("en")
print("Pipeline components:")
for name in nlp.pipe_names:
    component = nlp.get_pipe(name)
    print(f"  {name:40s}  ({type(component).__name__})")

Pipeline components:
  medspacy_pyrush                           (PyRuSHSentencizer)
  medspacy_target_matcher                   (TargetMatcher)
  medspacy_context                          (ConText)
  medspacy_sectionizer                      (Sectionizer)
  cwyde_category_mapper                     (CategoryMapperComponent)
  cwyde_indication_detector                 (IndicationDetectorComponent)
  cwyde_conflict_resolver                   (ConflictResolverComponent)
  cwyde_section_propagator                  (SectionPropagatorComponent)
  cwyde_consistency_checker                 (ConsistencyCheckerComponent)


The pipeline has two layers:

**medspaCy foundation** (sentence splitting → entity matching → context → section detection):

| Component | Role |
|---|---|
| `medspacy_pyrush` | PyRuSH rule-based sentence segmenter (clinical-optimised) |
| `medspacy_target_matcher` | Matches target concepts (findings, diagnoses) using `TargetRule`s |
| `medspacy_context` | Applies modifier rules: identifies negation, uncertainty, temporality, family scope |
| `medspacy_sectionizer` | Labels document sections (Impression, History, Assessment, …) |

**cwyde components** (lift medspaCy's booleans → rich assertion categories → formal semantics):

| Component | Role |
|---|---|
| `cwyde_category_mapper` | Maps medspaCy context booleans → `AssertionCategory` |
| `cwyde_indication_detector` | Promotes `DEFINITE_NEGATED_EXISTENCE` → `INDICATION` when section/cue context warrants it |
| `cwyde_conflict_resolver` | Resolves category conflicts on the same entity span (e.g. HYPOTHETICAL + DEFINITE_EXISTENCE → HYPOTHETICAL) |
| `cwyde_section_propagator` | Propagates section-level default assertions to entities with no explicit modifier |
| `cwyde_consistency_checker` | Optionally verifies final formulas for logical consistency via `gamen-validate` |

## 2. Adding target rules

Clinical findings are not built into the pipeline — they are domain-specific.
A `TargetRule` specifies a literal phrase (or regex) and a semantic type.

In [2]:
from medspacy.target_matcher import TargetRule

matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
])
print(f"Target matcher now recognises {len(matcher.rules)} target rule(s).")

Target matcher now recognises 2 target rule(s).


This is the cwyde equivalent of pyConTextNLP's `itemData` for *targets*.
In pyConTextNLP you read targets from a YAML file:

```python
targets = itemData.get_items("https://.../utah_crit.yml")
markup.markItems(targets, mode="target")
```

In cwyde the `medspacy_target_matcher` component handles this declaratively.
Modifier rules (negation, uncertainty, etc.) come from the knowledge base YAML
files and are loaded automatically by `cwyde.load()`.

## 3. Processing a single sentence

Let's start with the simplest case — a clear negation — and inspect every attribute cwyde sets.

In [3]:
sentence = "No evidence of pulmonary embolism."
doc = nlp(sentence)

ent = doc.ents[0]
print(f"Entity:  {ent.text!r}  [{ent.start_char}:{ent.end_char}]")
print()
print("medspaCy boolean context attributes:")
print(f"  is_negated:    {ent._.is_negated}")
print(f"  is_uncertain:  {ent._.is_uncertain}")
print(f"  is_historical: {ent._.is_historical}")
print(f"  is_family:     {ent._.is_family}")
print()
print("cwyde assertion attributes:")
print(f"  cwyde_assertion_category: {ent._.cwyde_assertion_category.value}")
print(f"  cwyde_belief_agent:       {ent._.cwyde_belief_agent}")
print(f"  cwyde_modal_formula:      {ent._.cwyde_modal_formula}")
print(f"  cwyde_consistent:         {ent._.cwyde_consistent}")
print(f"  cwyde_section_inherited:  {ent._.cwyde_section_inherited}")

Entity:  'pulmonary embolism'  [15:33]

medspaCy boolean context attributes:
  is_negated:    True
  is_uncertain:  False
  is_historical: False
  is_family:     False

cwyde assertion attributes:
  cwyde_assertion_category: DEFINITE_NEGATED_EXISTENCE
  cwyde_belief_agent:       clinician
  cwyde_modal_formula:      RankedBelief(agent='clinician', rank=-2, operand=Atom(name='pulmonary embolism'))
  cwyde_consistent:         True
  cwyde_section_inherited:  False


What happened at each pipeline stage?

1. **`medspacy_target_matcher`** — matched `"pulmonary embolism"` as a `CONDITION` entity.
2. **`medspacy_context`** — identified `"no evidence of"` as a `DEFINITE_NEGATED_EXISTENCE`
   modifier firing *forward*; set `is_negated = True`.
3. **`cwyde_category_mapper`** — mapped `is_negated=True, is_uncertain=False` →
   `DEFINITE_NEGATED_EXISTENCE`.  
   (medspaCy cannot distinguish definite from probable negation; cwyde does so via
   the specific modifier literal.)
4. **`cwyde_modal_formula`** — v0.3 encoding: `RankedBelief(clinician, -2, Atom("pulmonary embolism"))`,
   the clinician's two-sided OCF rank for this finding is −2 (firm disbelief).

### Inspecting the resolution trace

Each entity carries a `cwyde_resolution_trace` recording the category decision at every step — the cwyde equivalent of inspecting the ConText graph edges in pyConTextNLP.

In [4]:
trace = ent._.cwyde_resolution_trace
print("Resolution trace (one entry per pipeline component that acted):")
for step in trace:
    ms  = str(step.get("medspacy", "—"))
    cwy = str(step.get("cwyde", "—"))
    print(f"  step={step['step']!r:25s}  medspacy={ms:30s}  →  cwyde={cwy}")

Resolution trace (one entry per pipeline component that acted):
  step='category_mapper'          medspacy=NEGATED_EXISTENCE               →  cwyde=AssertionCategory.DEFINITE_NEGATED_EXISTENCE


## 4. The knowledge base

cwyde keeps all clinical knowledge in **YAML files** — the direct successor to
pyConTextNLP's tab-separated itemData. There are four kinds:

| File | Role |
|---|---|
| `core/categories.yaml` | Defines the nine `AssertionCategory` values and their formal semantics |
| `lang/en/lexicon/*.yaml` | Modifier literals: negation phrases, uncertainty hedges, temporal cues, etc. |
| `core/section_assertions.yaml` | Default assertion per document section (e.g. History → HISTORICAL) |
| `core/interaction_rules.yaml` | Conflict resolution rules (which category overrides which) |

This separates clinical knowledge from code — Buchanan's separation principle — making the
KB auditable and extendable without touching Python.

In [5]:
import yaml
import cwyde_knowledge

data_root = cwyde_knowledge.data_root()
cats_data  = yaml.safe_load(open(data_root / "core/categories.yaml"))

print(f"KB schema_version: {cats_data['schema_version']}\n")
print(f"{'Category':35s}  {'Axis':12s}  Modal reading")
print("-" * 105)
for c in cats_data["categories"]:
    if c["category"] == "UNRESOLVED":
        continue
    print(f"  {c['category']:33s}  {c.get('axis','—'):12s}  {c['modal_reading'][:55]}")

KB schema_version: 3

Category                             Axis          Modal reading
---------------------------------------------------------------------------------------------------------
  DEFINITE_EXISTENCE                 existence     τ_clinician(X) = 2 — clinician believes X present at fi
  PROBABLE_EXISTENCE                 existence     τ_clinician(X) = 1 — clinician believes X present at fi
  AMBIVALENT_EXISTENCE               existence     τ_clinician(X) = 0 — clinician explicitly neutral; neit
  PROBABLE_NEGATED_EXISTENCE         existence     τ_clinician(X) = -1 — clinician disbelieves X at firmne
  DEFINITE_NEGATED_EXISTENCE         existence     τ_clinician(X) = -2 — clinician disbelieves X at firmne
  INDICATION                         epistemic     ¬K_clinician(X) ∧ ¬K_clinician(¬X) — clinician investig
  HISTORICAL                         temporality   B_clinician(P(X)) — clinician believes X was the case
  HYPOTHETICAL                       temporality   B_clinici

### Modifier lexicon

The modifier lexicon is the cwyde equivalent of pyConTextNLP's `itemData` for *modifiers*.
Each entry has a `lex` (phrase or regex), a `category`, and a `direction`
(`forward`, `backward`, or `bidirectional`).

In pyConTextNLP you loaded modifiers from a URL:
```python
modifiers = itemData.get_items("https://.../lexical_kb_05042016.yml")
markup.markItems(modifiers, mode="modifier")
```

In cwyde the YAML lexicon files are loaded automatically at pipeline construction time.

In [6]:
from collections import defaultdict

lex_dir = data_root / "lang/en/lexicon"
by_cat  = defaultdict(list)

for f in sorted(lex_dir.glob("*.yaml")):
    data = yaml.safe_load(open(f))
    # files have either a top-level list or {"entries": [...]} wrapper
    entries = data.get("entries", data) if isinstance(data, dict) else data
    if not isinstance(entries, list):
        continue
    for entry in entries:
        cat     = entry.get("category", "")
        literal = entry.get("lex", entry.get("literal", ""))
        if cat and literal:
            by_cat[cat].append(literal)

total = sum(len(v) for v in by_cat.values())
print(f"Total modifier entries across all lexicon files: {total}\n")
print(f"Sample literals by category (up to 4 each):")
for cat in ["DEFINITE_NEGATED_EXISTENCE", "PROBABLE_NEGATED_EXISTENCE",
            "PROBABLE_EXISTENCE", "AMBIVALENT_EXISTENCE",
            "HISTORICAL", "INDICATION", "FAMILY"]:
    examples = by_cat.get(cat, [])[:4]
    print(f"  {cat:35s}: {examples}")

Total modifier entries across all lexicon files: 1111

Sample literals by category (up to 4 each):
  DEFINITE_NEGATED_EXISTENCE         : ['no', ': no', ': none', 'no evidence of']
  PROBABLE_NEGATED_EXISTENCE         : ['no XYZ to suggest', 'unlikely', 'most likely not', 'probably not']
  PROBABLE_EXISTENCE                 : ['probable', 'probably', 'likely', 'possible']
  AMBIVALENT_EXISTENCE               : ['cannot exclude', 'cannot be excluded', 'equivocal', 'indeterminate']
  HISTORICAL                         : ['history of', 'prior', 'previous', 'known']
  INDICATION                         : ['rule out', 'r/o', 'evaluate for', 'to assess for']
  FAMILY                             : ['family history of', 'father', 'mother', 'sibling']


## 5. All nine assertion categories

The nine `AssertionCategory` values cover the full assertion space for clinical findings.
The sentences below are representative PE radiology report examples — the same domain
as the original pyConTextNLP peFinder paper.

In [7]:
examples = {
    "DEFINITE_EXISTENCE":         "Pulmonary embolism is present.",
    "PROBABLE_EXISTENCE":         "Probable pulmonary embolism in the right lower lobe.",
    "AMBIVALENT_EXISTENCE":       "Cannot exclude pulmonary embolism.",
    "PROBABLE_NEGATED_EXISTENCE": "Pulmonary embolism is unlikely.",
    "DEFINITE_NEGATED_EXISTENCE": "No evidence of pulmonary embolism.",
    "HISTORICAL":                 "History of pulmonary embolism.",
    "INDICATION":                 "Rule out pulmonary embolism.",
    "FAMILY":                     "Patient's mother had pulmonary embolism.",
}

print(f"{'Expected':35s}  {'Got':35s}  {'OK':2s}  Text")
print("-" * 115)
for expected, text in examples.items():
    doc = nlp(text)
    for ent in doc.ents:
        got = ent._.cwyde_assertion_category.value
        ok  = "✓" if got == expected else "✗"
        print(f"  {expected:33s}  {got:35s}  {ok}   {text}")

Expected                             Got                                  OK  Text
-------------------------------------------------------------------------------------------------------------------


  DEFINITE_EXISTENCE                 DEFINITE_EXISTENCE                   ✓   Pulmonary embolism is present.


  PROBABLE_EXISTENCE                 PROBABLE_EXISTENCE                   ✓   Probable pulmonary embolism in the right lower lobe.
  AMBIVALENT_EXISTENCE               AMBIVALENT_EXISTENCE                 ✓   Cannot exclude pulmonary embolism.


  PROBABLE_NEGATED_EXISTENCE         PROBABLE_NEGATED_EXISTENCE           ✓   Pulmonary embolism is unlikely.


  DEFINITE_NEGATED_EXISTENCE         DEFINITE_NEGATED_EXISTENCE           ✓   No evidence of pulmonary embolism.


  HISTORICAL                         HISTORICAL                           ✓   History of pulmonary embolism.
  INDICATION                         INDICATION                           ✓   Rule out pulmonary embolism.


  FAMILY                             FAMILY                               ✓   Patient's mother had pulmonary embolism.


## 6. medspaCy booleans vs. cwyde categories

medspaCy provides four boolean attributes (`is_negated`, `is_uncertain`, `is_historical`,
`is_family`) derived from ConText. These are useful but **collapse clinically meaningful
distinctions**:

- `is_negated = True` covers both *definitely absent* and *probably absent* — cwyde
  distinguishes `DEFINITE_NEGATED_EXISTENCE` (rank −2) from `PROBABLE_NEGATED_EXISTENCE` (rank −1).
- `is_uncertain = True` covers both *probably present* and *equivocal* — cwyde
  distinguishes `PROBABLE_EXISTENCE` (rank +1) from `AMBIVALENT_EXISTENCE` (rank 0).
- `INDICATION` (rule-out context) maps to `is_negated = False` with no further signal
  in medspaCy. cwyde returns `INDICATION` — a categorically different epistemic state:
  the clinician does not yet know whether the finding is present.

In [8]:
sentences = list(examples.values())

col = [44, 12, 14, 15, 11, 30]
headers = ["Sentence", "is_negated", "is_uncertain", "is_historical", "is_family", "cwyde_category"]
print("".join(h.ljust(col[i]) for i, h in enumerate(headers)))
print("-" * sum(col))

for text in sentences:
    doc = nlp(text)
    for ent in doc.ents:
        row = [
            text[:42],
            str(ent._.is_negated),
            str(ent._.is_uncertain),
            str(ent._.is_historical),
            str(ent._.is_family),
            ent._.cwyde_assertion_category.value,
        ]
        print("".join(v.ljust(col[i]) for i, v in enumerate(row)))

Sentence                                    is_negated  is_uncertain  is_historical  is_family  cwyde_category                
------------------------------------------------------------------------------------------------------------------------------


Pulmonary embolism is present.              False       False         False          False      DEFINITE_EXISTENCE            


Probable pulmonary embolism in the right l  False       False         False          False      PROBABLE_EXISTENCE            
Cannot exclude pulmonary embolism.          False       False         False          False      AMBIVALENT_EXISTENCE          


Pulmonary embolism is unlikely.             False       False         False          False      PROBABLE_NEGATED_EXISTENCE    


No evidence of pulmonary embolism.          True        False         False          False      DEFINITE_NEGATED_EXISTENCE    


History of pulmonary embolism.              False       False         True           False      HISTORICAL                    
Rule out pulmonary embolism.                False       False         False          False      INDICATION                    


Patient's mother had pulmonary embolism.    False       False         False          True       FAMILY                        


Note:
- Row 4 (`Pulmonary embolism is unlikely`): `is_negated=True` in medspaCy, but cwyde
  returns `PROBABLE_NEGATED_EXISTENCE` (rank −1) — distinct from
  `DEFINITE_NEGATED_EXISTENCE` (rank −2, row 5).
- Row 7 (`Rule out pulmonary embolism`): `is_negated=False` in medspaCy with no further
  signal. cwyde returns `INDICATION` — open epistemic question, not negation.

## 7. The modal formula (v0.3 Spohn OCF encoding)

Each `AssertionCategory` is mapped to a doxastic formula using
[Spohn's (1988) ordinal conditional functions](https://doi.org/10.1007/978-94-009-2865-7_6)
(OCF / ranking theory). The signed integer `rank` encodes the clinician's two-sided
doxastic rank τ for the finding:

| rank | Interpretation |
|---:|---|
| +2 | Firmly believed (DEFINITE_EXISTENCE) |
| +1 | Probably believed (PROBABLE_EXISTENCE) |
|  0 | Neutral — neither believed nor disbelieved (AMBIVALENT_EXISTENCE) |
| −1 | Probably disbelieved (PROBABLE_NEGATED_EXISTENCE) |
| −2 | Firmly disbelieved (DEFINITE_NEGATED_EXISTENCE) |

HISTORICAL and FAMILY use binary `Belief` (KD45 doxastic logic);
INDICATION uses the open-question encoding `¬K_a(X) ∧ ¬K_a(¬X)`.

In [9]:
print(f"{'Category':35s}  {'rank':>5}  Formula")
print("-" * 95)
for expected, text in examples.items():
    doc = nlp(text)
    for ent in doc.ents:
        f    = ent._.cwyde_modal_formula
        rank = getattr(f, "rank", "—")
        print(f"  {expected:33s}  {str(rank):>5}  {f}")

Category                              rank  Formula
-----------------------------------------------------------------------------------------------


  DEFINITE_EXISTENCE                     2  RankedBelief(agent='clinician', rank=2, operand=Atom(name='Pulmonary embolism'))


  PROBABLE_EXISTENCE                     1  RankedBelief(agent='clinician', rank=1, operand=Atom(name='pulmonary embolism'))
  AMBIVALENT_EXISTENCE                   0  RankedBelief(agent='clinician', rank=0, operand=Atom(name='pulmonary embolism'))


  PROBABLE_NEGATED_EXISTENCE            -1  RankedBelief(agent='clinician', rank=-1, operand=Atom(name='Pulmonary embolism'))


  DEFINITE_NEGATED_EXISTENCE            -2  RankedBelief(agent='clinician', rank=-2, operand=Atom(name='pulmonary embolism'))


  HISTORICAL                             —  Belief(agent='clinician', operand=Past(operand=Atom(name='pulmonary embolism')))
  INDICATION                             —  Indication(operand=Atom(name='pulmonary embolism'), agent='clinician')


  FAMILY                                 —  Belief(agent='clinician', operand=Atom(name='pulmonary embolism_family'))


### Wire format for gamen-validate

The formula is serialised to JSON tree format and sent to `gamen-validate` for
consistency checking. This is what travels over the stdin/stdout bridge to the
Haskell prover — the same protocol used by `guideline-validation`.

In [10]:
import json

# Show the wire format for two formulas that gamen-validate would check for consistency
sentence_a = "Probable pulmonary embolism in the right lower lobe."  # rank +1
sentence_b = "No PE identified on the left side."                     # rank -2

for text in [sentence_a, sentence_b]:
    doc = nlp(text)
    for ent in doc.ents:
        wire = ent._.cwyde_modal_formula.to_tree_json()
        print(f"# {text}")
        print(json.dumps(wire, indent=2))
        print()

# Probable pulmonary embolism in the right lower lobe.
{
  "type": "ranked_belief",
  "agent": "clinician",
  "rank": 1,
  "operand": {
    "type": "atom",
    "name": "pulmonary embolism"
  }
}



# No PE identified on the left side.
{
  "type": "ranked_belief",
  "agent": "clinician",
  "rank": -2,
  "operand": {
    "type": "atom",
    "name": "PE"
  }
}



## Takeaways

1. **Nine categories, not four booleans.** cwyde's `AssertionCategory` preserves the
   DEFINITE/PROBABLE split on both the positive and negative sides, and adds INDICATION
   as a first-class category — three distinctions invisible to medspaCy's booleans.

2. **Declarative KB.** Modifier rules live in YAML files (`lang/en/lexicon/*.yaml`),
   not in code. Adding a new negation phrase is a one-line YAML edit, not a code change.

3. **Formal semantics included.** Every assertion is simultaneously a Python enum
   (`AssertionCategory`) and a logical formula (`RankedBelief`, `Belief`, `Indication`)
   that can be reasoned over by `gamen-validate` — enabling consistency checking and
   downstream guideline validation.

4. **Traceable decisions.** `ent._.cwyde_resolution_trace` records every step in the
   category decision, making the system auditable and debuggable.

**Next notebooks:**
- `02_indication_detection.ipynb` — deep dive on the INDICATION category
- `03_conflict_resolution.ipynb` — handling conflicting modifiers
- `04_section_propagation.ipynb` — document-level section defaults
- `05_with_gamen.ipynb` — consistency checking via the gamen-validate bridge